In [16]:
import sys
sys.path.insert(0, "/home/anubhav/development/vla_from_scratch")  # your project root

import torch
from torch.utils.flop_counter import FlopCounterMode
from vla_diffusion import VlaDiffusion
from utils.tokenizer import tokenize_instruction

In [17]:
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
STATE_DIM  = 39
ACTION_DIM = 4
D_MODEL    = 256
B          = 64

model = VlaDiffusion(state_dim=STATE_DIM, action_dim=ACTION_DIM, d_model=D_MODEL).to(DEVICE)
# or load your trained checkpoint
# ckpt = torch.load("checkpoints/best.pt", map_location=DEVICE)
# model.load_state_dict(ckpt["model"])
model.eval()

image  = torch.randn(B, 3, 224, 224).to(DEVICE)
state  = torch.randn(B, STATE_DIM).to(DEVICE)
action = torch.randn(B, ACTION_DIM).to(DEVICE)
ids, mask = tokenize_instruction("reach the target")
text_ids  = ids.repeat(B, 1).to(DEVICE)
text_mask = mask.repeat(B, 1).to(DEVICE)

# precompute intermediate tensors so lambdas don't double-count
with torch.no_grad():
    vis_tok   = model.vision_encoder(image)
    txt_tok   = model.text_encoder(text_ids, text_mask)
    state_tok = model.state_encoder(state)
    fused     = model.fusion(vis_tok, txt_tok, state_tok)

print("Setup done")


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7232.81it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setup done


In [18]:
print("── Param count ──")
for name, module in [
    ("vision_encoder", model.vision_encoder),
    ("text_encoder",   model.text_encoder),
    ("state_encoder",  model.state_encoder),
    ("fusion",         model.fusion),
    ("diffusion_head", model.diffusion_head),
]:
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    frozen    = sum(p.numel() for p in module.parameters() if not p.requires_grad)
    print(f"  {name:<20} trainable: {trainable:>9,}  frozen: {frozen:>9,}")
print(f"\n  {'TOTAL':<20} {sum(p.numel() for p in model.parameters()):>9,}")

── Param count ──
  vision_encoder       trainable: 10,625,280  frozen:   683,072
  text_encoder         trainable:   197,376  frozen: 66,362,880
  state_encoder        trainable:    38,656  frozen:         0
  fusion               trainable: 1,580,288  frozen:         0
  diffusion_head       trainable:    54,532  frozen:         0

  TOTAL                79,542,084


In [19]:
print("── FLOPs (batch=64) ──")

flop_fns = {
    "vision_encoder":      lambda: model.vision_encoder(image),
    "text_encoder":        lambda: model.text_encoder(text_ids, text_mask),
    "state_encoder":       lambda: model.state_encoder(state),
    "fusion":              lambda: model.fusion(vis_tok, txt_tok, state_tok),  # uses precomputed
    "diffusion_head.loss": lambda: model.diffusion_head.loss(action, fused),
}

for name, fn in flop_fns.items():
    with FlopCounterMode(display=False) as fc:
        fn()
    print(f"  {name:<25} {fc.get_total_flops()/1e9:.3f} GFLOPs")

── FLOPs (batch=64) ──
  vision_encoder            232.958 GFLOPs
  text_encoder              164.891 GFLOPs
  state_encoder             0.005 GFLOPs
  fusion                    17.167 GFLOPs
  diffusion_head.loss       0.007 GFLOPs


In [20]:
if DEVICE == "cuda":
    print("── Peak GPU memory ──")
    
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        model.loss(image, text_ids, text_mask, state, action)
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"  full forward+loss:     {peak:.2f} GB")

    for name, fn in [
        ("vision_encoder",  lambda: model.vision_encoder(image)),
        ("text_encoder",    lambda: model.text_encoder(text_ids, text_mask)),
        ("state_encoder",   lambda: model.state_encoder(state)),
        ("fusion",          lambda: model.fusion(vis_tok, txt_tok, state_tok)),
        ("diffusion_head",  lambda: model.diffusion_head.loss(action, fused)),
    ]:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            fn()
        mem = torch.cuda.max_memory_allocated() / 1024**3
        print(f"  {name:<25} {mem:.3f} GB")
else:
    print("No CUDA — skipping memory profiling")

── Peak GPU memory ──
  full forward+loss:     0.73 GB
  vision_encoder            0.733 GB
  text_encoder              0.411 GB
  state_encoder             0.350 GB
  fusion                    0.395 GB
  diffusion_head            0.350 GB


In [21]:
import time

def time_fn(fn, device, n=20):
    if device == "cuda":
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        for _ in range(3): fn()  # warmup
        torch.cuda.synchronize()
        start.record()
        for _ in range(n): fn()
        end.record()
        torch.cuda.synchronize()
        return start.elapsed_time(end) / n
    else:
        for _ in range(3): fn()
        t0 = time.perf_counter()
        for _ in range(n): fn()
        return (time.perf_counter() - t0) / n * 1000

print("── Wall time per module (avg 20 runs) ──")
with torch.no_grad():
    for name, fn in [
        ("vision_encoder",     lambda: model.vision_encoder(image)),
        ("text_encoder",       lambda: model.text_encoder(text_ids, text_mask)),
        ("state_encoder",      lambda: model.state_encoder(state)),
        ("fusion",             lambda: model.fusion(vis_tok, txt_tok, state_tok)),
        ("diffusion sampling", lambda: model.diffusion_head.sample(fused)),
    ]:
        ms = time_fn(fn, DEVICE)
        print(f"  {name:<25} {ms:.1f} ms")

── Wall time per module (avg 20 runs) ──
  vision_encoder            233.3 ms
  text_encoder              251.7 ms
  state_encoder             0.1 ms
  fusion                    22.4 ms
  diffusion sampling        4.2 ms


In [ ]:
# Cell 6 — updated param count with action_horizon
ACTION_HORIZON = 16

model_v2 = VlaDiffusion(state_dim=STATE_DIM, action_dim=ACTION_DIM,
                         d_model=D_MODEL, action_horizon=ACTION_HORIZON).to(DEVICE)
model_v2.eval()

total     = sum(p.numel() for p in model_v2.parameters())
trainable = sum(p.numel() for p in model_v2.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"{'Module':<20} {'Trainable':>12} {'Frozen':>12}")
print("-" * 46)
for name, module in [
    ("vision_encoder", model_v2.vision_encoder),
    ("text_encoder",   model_v2.text_encoder),
    ("state_encoder",  model_v2.state_encoder),
    ("fusion",         model_v2.fusion),
    ("diffusion_head", model_v2.diffusion_head),
]:
    t = sum(p.numel() for p in module.parameters() if p.requires_grad)
    f = sum(p.numel() for p in module.parameters() if not p.requires_grad)
    print(f"{name:<20} {t:>12,} {f:>12,}")
print("-" * 46)
print(f"{'TOTAL':<20} {total:>12,}  ({trainable:,} trainable, {frozen:,} frozen)")

In [ ]:
# Cell 7 — updated intermediates for v2 (action chunk shape changed)
action_chunk = torch.randn(B, ACTION_HORIZON, ACTION_DIM).to(DEVICE)  # (B, H, action_dim)

with torch.no_grad():
    vis_tok_v2   = model_v2.vision_encoder(image)
    txt_tok_v2   = model_v2.text_encoder(text_ids, text_mask)
    state_tok_v2 = model_v2.state_encoder(state)
    fused_v2     = model_v2.fusion(vis_tok_v2, txt_tok_v2, state_tok_v2)

print(f"vision tokens:  {vis_tok_v2.shape}")    # (B, 49, 256)
print(f"text tokens:    {txt_tok_v2.shape}")    # (B, L, 256)
print(f"state tokens:   {state_tok_v2.shape}")  # (B, 1, 256)
print(f"fused context:  {fused_v2.shape}")      # (B, 256)
print(f"action chunk:   {action_chunk.shape}")  # (B, 16, 4)

In [ ]:
# Cell 8 — FLOPs comparison v1 vs v2
print("── FLOPs comparison: single action vs action chunk ──")
print(f"{'Module':<25} {'v1 (single)':>14} {'v2 (chunk=16)':>14}")
print("-" * 55)

pairs = [
    ("vision_encoder",
        lambda: model.vision_encoder(image),
        lambda: model_v2.vision_encoder(image)),
    ("text_encoder",
        lambda: model.text_encoder(text_ids, text_mask),
        lambda: model_v2.text_encoder(text_ids, text_mask)),
    ("fusion",
        lambda: model.fusion(vis_tok, txt_tok, state_tok),
        lambda: model_v2.fusion(vis_tok_v2, txt_tok_v2, state_tok_v2)),
    ("diffusion_head",
        lambda: model.diffusion_head.loss(action, fused),
        lambda: model_v2.diffusion_head.loss(action_chunk, fused_v2)),
]

for name, fn1, fn2 in pairs:
    with FlopCounterMode(display=False) as fc: fn1()
    f1 = fc.get_total_flops() / 1e9
    with FlopCounterMode(display=False) as fc: fn2()
    f2 = fc.get_total_flops() / 1e9
    print(f"{name:<25} {f1:>12.3f}G {f2:>12.3f}G")

In [ ]:
# Cell 9 — wall time v2 with text cache effect
print("── Wall time v2 (avg 20 runs) ──")
print(f"{'Module':<25} {'time (ms)':>10}")
print("-" * 37)

with torch.no_grad():
    for name, fn in [
        ("vision_encoder",     lambda: model_v2.vision_encoder(image)),
        ("text_encoder",       lambda: model_v2.text_encoder(text_ids, text_mask)),
        ("text (cached)",      lambda: model_v2._text_cache.get(text_ids.sum().item())),  # ~0ms
        ("state_encoder",      lambda: model_v2.state_encoder(state)),
        ("fusion",             lambda: model_v2.fusion(vis_tok_v2, txt_tok_v2, state_tok_v2)),
        ("diffusion sampling", lambda: model_v2.diffusion_head.sample(fused_v2)),
    ]:
        ms = time_fn(fn, DEVICE)
        print(f"  {name:<25} {ms:>8.1f} ms")

In [ ]:
# Cell 10 — memory comparison v1 vs v2
if DEVICE == "cuda":
    print("── GPU memory: single action vs chunk ──")
    print(f"{'Pass':<30} {'Memory (GB)':>12}")
    print("-" * 44)

    for label, fn in [
        ("v1 full forward+loss",
            lambda: model.loss(image, text_ids, text_mask, state, action)),
        ("v2 full forward+loss (chunk)",
            lambda: model_v2.loss(image, text_ids, text_mask, state, action_chunk)),
        ("v2 sampling",
            lambda: model_v2.diffusion_head.sample(fused_v2)),
    ]:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            fn()
        mem = torch.cuda.max_memory_allocated() / 1024**3
        print(f"  {label:<30} {mem:>10.3f} GB")